### Phase 0 & Phase 1: Workspace Inspection and Lung Cancer Dataset Audit

In [17]:
%pip install --force-reinstall "numpy>=2.0,<2.3" "scikit-learn==1.9.0"

  Using cached numpy-2.2.6-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached scikit_learn-1.9.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.18.1-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.28.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached scikit_learn-1.9.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (9.1 MB)
Using cached numpy-2.2.6-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.5 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached narwhals-2.28.0-py3-none-any.whl (490 kB)
Using cached scipy-1.18.1-cp313-cp313-manylinux_2_27_x86_6

In [18]:
import os
import pandas as pd
import json

# 1. Print current workspace files
print("--- Files in Workspace ---")
files = os.listdir('.')
for f in files:
    if f.endswith('.ipynb') or f.endswith('.csv') or f.endswith('.json') or f.endswith('.pkl'):
        print(f)

# 2. Inspect the Diabetes reference implementation metadata if available
print("\n--- Checking Lung/Diabetes Metadata files ---")
metadata_file = 'lung_cancer_pipeline_metadata.json'
if os.path.exists(metadata_file):
    with open(metadata_file, 'r') as f:
        meta = json.load(f)
        print("Sample of current pipeline metadata structure:")
        print(json.dumps(meta, indent=2)[:500] + "... (truncated)")

--- Files in Workspace ---
lung_cancer_pipeline_metadata.json
01_EDA_Diabetes.ipynb
04_Model_Evaluation_Diabetes.ipynb
03_Model_Training_Diabetes.ipynb
survey lung cancer.csv
best_lung_cancer_pipeline.pkl
02_Preprocessing_Diabetes_ipynb.ipynb
diabetes_model.pkl
preprocessor.pkl

--- Checking Lung/Diabetes Metadata files ---
Sample of current pipeline metadata structure:
{
  "disease_name": "Lung Cancer",
  "target_column": "LUNG_CANCER",
  "feature_names": [
    "GENDER",
    "AGE",
    "SMOKING",
    "YELLOW_FINGERS",
    "ANXIETY",
    "PEER_PRESSURE",
    "CHRONIC DISEASE",
    "FATIGUE ",
    "ALLERGY ",
    "WHEEZING",
    "ALCOHOL CONSUMING",
    "COUGHING",
    "SHORTNESS OF BREATH",
    "SWALLOWING DIFFICULTY",
    "CHEST PAIN"
  ],
  "feature_types": {
    "GENDER": "int64",
    "AGE": "int64",
    "SMOKING": "int64",
    "YELLOW_FINGERS": "int64",
   ... (truncated)


In [19]:
# 3. Load and Audit the Lung Cancer Dataset
csv_path = 'survey lung cancer.csv'
if os.path.exists(csv_path):
    df_raw = pd.read_csv(csv_path)
    print(f"Dataset Shape: {df_raw.shape}")
    print("\nColumns and Data Types:")
    print(df_raw.dtypes)
    print("\nMissing Values:")
    print(df_raw.isnull().sum())
    print("\nTarget Distribution:")
    if 'LUNG_CANCER' in df_raw.columns:
        print(df_raw["LUNG_CANCER"].value_counts(dropna=False))
    print("\nFirst 5 Rows:")
    display(df_raw.head())
else:
    print(f"{csv_path} not found in the workspace!")

Dataset Shape: (309, 16)

Columns and Data Types:
GENDER                   object
AGE                       int64
SMOKING                   int64
YELLOW_FINGERS            int64
ANXIETY                   int64
PEER_PRESSURE             int64
CHRONIC DISEASE           int64
FATIGUE                   int64
ALLERGY                   int64
WHEEZING                  int64
ALCOHOL CONSUMING         int64
COUGHING                  int64
SHORTNESS OF BREATH       int64
SWALLOWING DIFFICULTY     int64
CHEST PAIN                int64
LUNG_CANCER              object
dtype: object

Missing Values:
GENDER                   0
AGE                      0
SMOKING                  0
YELLOW_FINGERS           0
ANXIETY                  0
PEER_PRESSURE            0
CHRONIC DISEASE          0
FATIGUE                  0
ALLERGY                  0
WHEEZING                 0
ALCOHOL CONSUMING        0
COUGHING                 0
SHORTNESS OF BREATH      0
SWALLOWING DIFFICULTY    0
CHEST PAIN               0
LU

,GENDER,AGE,SMOKING,YELLOW_FINGERS,ANXIETY,PEER_PRESSURE,CHRONIC DISEASE,FATIGUE,ALLERGY,WHEEZING,ALCOHOL CONSUMING,COUGHING,SHORTNESS OF BREATH,SWALLOWING DIFFICULTY,CHEST PAIN,LUNG_CANCER
0,M,69,1,2,2,1,1,2,1,2,2,2,2,2,2,YES
1,M,74,2,1,1,1,2,2,2,1,1,1,2,2,2,YES
2,F,59,1,1,1,2,1,2,1,2,1,2,2,1,2,NO
3,M,63,2,2,2,1,1,1,1,1,2,1,1,2,2,NO
4,F,63,1,2,1,1,1,1,1,2,1,2,2,1,1,NO


### Phase 4: Corrected Production Training, Dynamic Selection & Evaluation
We will now run a clean, correct evaluation loop. We will calculate the true `balanced_accuracy_score` and `roc_auc_score` (using prediction probabilities) and dynamically select the champion model based on the cross-validation score.

In [20]:
import numpy as np
import pandas as pd
import pickle
import json
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef, confusion_matrix

# 1. Load data and verify properties
df_raw = pd.read_csv('survey lung cancer.csv')
print(f"Raw dataset row count: {len(df_raw)}")

# 2. Auditing duplicates
exact_dupes = df_raw.duplicated().sum()
print(f"Exact identical row count: {exact_dupes}")

# Apply policy: drop exact duplicates to prevent leakage across validation folds
exact_dupes = int(df_raw.duplicated().sum())

print("Dataset rows:", len(df_raw))
print("Exact duplicate rows:", exact_dupes)
print("Duplicate rows with conflicting targets:",
      int(df_raw.groupby(list(df_raw.columns), dropna=False)
          ["LUNG_CANCER"].nunique().gt(1).sum()))

# Retain rows unless the source confirms that these are
# duplicate records rather than distinct survey respondents.
df_clean = df_raw.copy().reset_index(drop=True)
print(f"Cleaned dataset row count: {len(df_clean)}")

# 3. Apply standard clinical/survey feature transformations
gender_map = {'M': 1, 'F': 0}
target_map = {'YES': 1, 'NO': 0}
survey_map = {1: 0, 2: 1}

df_processed = df_clean.copy()
df_processed['GENDER'] = df_processed['GENDER'].map(gender_map)
df_processed['LUNG_CANCER'] = df_processed['LUNG_CANCER'].map(target_map)

survey_cols = [col for col in df_processed.columns if col not in ['AGE', 'GENDER', 'LUNG_CANCER']]
for col in survey_cols:
    df_processed[col] = df_processed[col].map(survey_map)

# Define features and target
X = df_processed.drop(columns=['LUNG_CANCER'])
y = df_processed['LUNG_CANCER']

# Stratified split to ensure identical minority class ratio in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Compare models using Stratified 5-Fold Cross Validation
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', random_state=42),
    'Support Vector Machine': SVC(class_weight='balanced', probability=True, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', model)
    ])

    # Use correct cross_validate scoring metrics
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=['balanced_accuracy', 'f1', 'roc_auc'])
    cv_results[name] = {
        'Mean Balanced Accuracy': np.mean(scores['test_balanced_accuracy']),
        'Mean F1': np.mean(scores['test_f1']),
        'Mean ROC-AUC': np.mean(scores['test_roc_auc'])
    }

print("\n--- Cross Validation Results on Train Set ---")
for name, metrics in cv_results.items():
    print(f"{name}: Balanced Acc = {metrics['Mean Balanced Accuracy']:.4f}, F1 = {metrics['Mean F1']:.4f}, ROC-AUC = {metrics['Mean ROC-AUC']:.4f}")

# 5. Dynamically pick best model based on Mean Balanced Accuracy
best_model_name = max(cv_results, key=lambda k: cv_results[k]['Mean Balanced Accuracy'])
print(f"\nSelected Champion Model: {best_model_name}")

# 6. Fit selected model on whole train set and run inference on test set
champion_model = models[best_model_name]
champion_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', champion_model)
])

champion_pipeline.fit(X_train, y_train)
y_pred = champion_pipeline.predict(X_test)
y_pred_proba = champion_pipeline.predict_proba(X_test)[:, 1]

# Compute exact test metrics using separate sklearn calls
test_metrics = {
    'accuracy': float(accuracy_score(y_test, y_pred)),
    'balanced_accuracy': float(balanced_accuracy_score(y_test, y_pred)),
    'precision': float(precision_score(y_test, y_pred)),
    'recall': float(recall_score(y_test, y_pred)),
    'f1_score': float(f1_score(y_test, y_pred)),
    'roc_auc': float(roc_auc_score(y_test, y_pred_proba)),
    'mcc': float(matthews_corrcoef(y_test, y_pred))
}

print("\n--- Final Test Set Evaluation (Unseen Data) ---")
for metric, score in test_metrics.items():
    print(f"{metric}: {score:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Raw dataset row count: 309
Exact identical row count: 33
Dataset rows: 309
Exact duplicate rows: 33
Duplicate rows with conflicting targets: 0
Cleaned dataset row count: 309

--- Cross Validation Results on Train Set ---
Logistic Regression: Balanced Acc = 0.8111, F1 = 0.9174, ROC-AUC = 0.9140
Random Forest: Balanced Acc = 0.7100, F1 = 0.9454, ROC-AUC = 0.8473
Support Vector Machine: Balanced Acc = 0.7419, F1 = 0.9171, ROC-AUC = 0.8773

Selected Champion Model: Logistic Regression

--- Final Test Set Evaluation (Unseen Data) ---
accuracy: 0.8710
balanced_accuracy: 0.9259
precision: 1.0000
recall: 0.8519
f1_score: 0.9200
roc_auc: 0.9537
mcc: 0.6526

Confusion Matrix:
[[ 8  0]
 [ 8 46]]


### Phase 5: Exporting Complete Metadata & Artifact Reload Validation
We now export our model pipeline along with production metadata containing exact feature schema ordering, feature value maps, and classification targets.

In [21]:
# 1. Save pipeline model
pipeline_filename = 'best_lung_cancer_pipeline.pkl'
with open(pipeline_filename, 'wb') as f:
    pickle.dump(champion_pipeline, f)

# 2. Save complete compatible metadata
metadata = {
  "disease_name": "Lung Cancer",
  "target_column": "LUNG_CANCER",
  "feature_names": X_train.columns.tolist(),
  "feature_types": {col: str(X_train[col].dtype) for col in X_train.columns},
  "target_mapping": {"YES": 1, "NO": 0},
  "categorical_value_mappings": {
      "GENDER": {"M": 1, "F": 0},
      "survey_questions": {"NO": 0, "YES": 1}
  },
  "dataset_source": "Kaggle Survey Lung Cancer Dataset",
  "dataset_license": "CC0: Public Domain",
  "duplicate_policy": "Removed exact row duplicates to protect validation split integrity",
  "selected_model": best_model_name,
  "evaluation_metrics": test_metrics,
  "validation_method": "5-Fold Stratified Cross-Validation",
  "model_artifact": pipeline_filename,
  "metadata_version": "1.0.0"
}

metadata_filename = 'lung_cancer_pipeline_metadata.json'
with open(metadata_filename, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Saved {pipeline_filename} and {metadata_filename}.")

# 3. Verification & Reload Audit
with open(pipeline_filename, 'rb') as f:
    loaded_pipeline = pickle.load(f)

# Run a test row prediction
sample_row = X_test.iloc[[0]]
pred_val = loaded_pipeline.predict(sample_row)[0]
pred_prob = loaded_pipeline.predict_proba(sample_row)[0][1]
print(f"\nReload validation check successful!")
print(f"Sample Input Age: {sample_row['AGE'].values[0]}, predicted target: {pred_val} with probability {pred_prob:.4f}")

Saved best_lung_cancer_pipeline.pkl and lung_cancer_pipeline_metadata.json.

Reload validation check successful!
Sample Input Age: 76, predicted target: 1 with probability 0.9962
